# Auditoría de las seis fuentes

## tl;dr

Las fuentes contienen **220.031 registros** de seis ciudades. Su identidad se valida
mediante nombre, cabecera, bytes, filas y SHA-256 antes de cualquier análisis. La
procedencia, licencia, moneda y fecha de extracción permanecen desconocidas.

## Contexto y métodos

### Supuestos clave

Cada fila representa un anuncio dentro de su ciudad. Se comprueba la clave candidata
`city_key + id`, pero no se presupone que `id` sea global. Los outliers válidos se
conservan y se señalan mediante IQR; no se limpian de forma silenciosa.

### 1. Cargar evidencia de inventario y calidad

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
ARTIFACTS = Path(os.environ.get("AIRBNB_SUPPLY_ARTIFACTS_DIR", ROOT / "artifacts"))
inventory_path = ARTIFACTS / "quality/source-inventory.json"
inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
profile = pd.read_parquet(ARTIFACTS / "quality/source-profile.parquet")
findings = pd.read_parquet(ARTIFACTS / "quality/findings.parquet")
inventory_table = pd.DataFrame(inventory["sources"])[
    ["city_key", "file_name", "parsed_row_count", "byte_size", "identity_status"]
]
inventory_table

**Conclusión.** La suma esperada es 220.031 y todos los archivos deben aparecer como
`identity_verified`. Esta comprobación respalda la integridad técnica de la copia, no la
autoridad ni la actualidad de la fuente.

### 2. Revisar completitud y hallazgos por ciudad

In [ ]:
null_summary = (
    profile.query("null_count > 0")
    .sort_values(["null_rate", "source_id"], ascending=[False, True])
    [["source_id", "field", "row_count", "null_count", "null_rate"]]
)
open_findings = findings.query("failed_count > 0")[
    ["source_id", "check_id", "severity", "failed_count", "failed_rate", "impact"]
].sort_values(["severity", "failed_rate"], ascending=[True, False])
display(null_summary.head(30))
display(open_findings.head(30))

**Conclusión.** Los nulos y valores extremos se cuantifican por fuente y campo. Los
faltantes estructurales de columnas se distinguirán de un valor cero; los outliers se
retienen para análisis robusto. La utilidad de cada métrica depende de estas tasas y no
solo del volumen total.

## Takeaways

La base es apta para construir un modelo canónico siempre que se preserve el linaje y se
mantengan explícitas las ausencias. No permite inferir actividad reciente, divisa,
reservas ni representatividad del mercado completo.